# Lab 1 — Write your first skill (EP14)

The theory video said tools let an agent **act**, while a skill tells it **how a job should be done**.
In this lab we write a `paper-summary` skill as a folder, then compare the same task without and with the skill

**What you'll do**
1. A simple agent loop with paper-search tools (the loop from §22.3.4)
2. Give it the task **without a skill** and look at the result
3. Write the skill as a folder: `SKILL.md` (instructions · workflow · guardrails) + `example.md` (knowledge)
4. **Load** and **activate** the skill: add its prompt and pick the tools it needs
5. Check the guardrails in code, and retry when they fail (recovery)
6. Compare the results, and count how many tokens the skill adds

The paper records are teaching data (the arXiv ids, titles, authors and years are real; the one-line summaries are written for the lab); the model outputs come from real runs.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §22.1, §22.3.4, §22.4 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **5 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep14-lab1-write-skill.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs less than 0.1 US cents) |
| Packages | `openai`, `python-dotenv`, `pyyaml` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv pyyaml

In [ ]:
import json, os, re
from pathlib import Path
import yaml
from openai import OpenAI

In [ ]:
def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

## 1. The paper library, the tools, and the agent loop

A simulated library of eight papers (on agent memory and tool use), and three tools:

| tool | What it does |
|---|---|
| `search_papers(query)` | searches words in titles and summaries, returns ids and titles |
| `get_paper(paper_id)` | returns one paper's full record |
| `word_count(text)` | counts words |

In [ ]:
PAPERS = {  # teaching data: real arXiv ids, titles, authors and years; one-line summaries written for this lab
    "2310.08560": {"title": "MemGPT: Towards LLMs as Operating Systems", "authors": "Packer et al.", "year": 2023,
                   "summary": "the LLM pages facts between a small main context and a large external store, like an operating system's virtual memory"},
    "2304.03442": {"title": "Generative Agents: Interactive Simulacra of Human Behavior", "authors": "Park et al.", "year": 2023,
                   "summary": "agents keep a memory stream of observations, score memories by recency, importance and relevance, and write reflections"},
    "2303.11366": {"title": "Reflexion: Language Agents with Verbal Reinforcement Learning", "authors": "Shinn et al.", "year": 2023,
                   "summary": "after a failed try the agent writes a short lesson to its memory and uses it on the next try"},
    "2504.19413": {"title": "Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory", "authors": "Chhikara et al.", "year": 2025,
                   "summary": "an LLM extracts facts from each conversation and decides to add, update or delete them in a long-term store"},
    "2502.12110": {"title": "A-MEM: Agentic Memory for LLM Agents", "authors": "Xu et al.", "year": 2025,
                   "summary": "each memory is a linked note with keywords and tags, and new notes update links to old ones"},
    "2305.16291": {"title": "Voyager: An Open-Ended Embodied Agent with Large Language Models", "authors": "Wang et al.", "year": 2023,
                   "summary": "a Minecraft agent that stores verified code as a growing skill library and reuses it"},
    "2210.03629": {"title": "ReAct: Synergizing Reasoning and Acting in Language Models", "authors": "Yao et al.", "year": 2023,
                   "summary": "the model alternates reasoning steps and tool actions in one loop"},
    "2302.04761": {"title": "Toolformer: Language Models Can Teach Themselves to Use Tools", "authors": "Schick et al.", "year": 2023,
                   "summary": "a model learns by itself where to insert API calls in text"},
}

def search_papers(query: str, max_results: int = 5) -> str:
    words = [w for w in re.findall(r"[a-z]+", query.lower()) if len(w) > 2]
    hits = sorted(((sum(w in (p["title"] + " " + p["summary"]).lower() for w in words), pid)
                   for pid, p in PAPERS.items()), reverse=True)
    return json.dumps([{"id": pid, "title": PAPERS[pid]["title"]} for score, pid in hits[:max_results] if score])

def get_paper(paper_id: str) -> str:
    if paper_id not in PAPERS:
        return json.dumps({"error": f"unknown id {paper_id}; call search_papers first"})
    return json.dumps({"id": paper_id, **PAPERS[paper_id]})

def word_count(text: str) -> str:
    return json.dumps({"words": len(text.split())})

FUNCS = {"search_papers": search_papers, "get_paper": get_paper, "word_count": word_count}
SPECS = {
    "search_papers": {"type": "function", "function": {"name": "search_papers",
        "description": "Search the paper library by keywords in the title or summary. Returns ids and titles.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"},
                       "max_results": {"type": "integer"}}, "required": ["query"]}}},
    "get_paper": {"type": "function", "function": {"name": "get_paper",
        "description": "Get one paper's full record (title, authors, year, summary) by its arXiv id.",
        "parameters": {"type": "object", "properties": {"paper_id": {"type": "string"}}, "required": ["paper_id"]}}},
    "word_count": {"type": "function", "function": {"name": "word_count",
        "description": "Count the words in a text.",
        "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}},
}

**Model:** `google/gemma-4-26b-a4b-it` — the same model as every episode, pinned to Makora
· this lab needs **tool calling**, which Gemma 4 can do
· USD 0.09 per million input tokens and USD 0.28 per million output tokens — one summary run costs about 0.03 cents
· The `:free` version doesn't work (tested 2026-11-26: 4 out of 4 calls got 429 "rate-limited")

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}

In [ ]:
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0}

In [ ]:
def run_agent(system, task, tools, max_steps=8):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    info = {"prompt_tokens": None, "fetched": set()}
    for _ in range(max_steps):
        r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA, messages=msgs,
                                           tools=[SPECS[t] for t in tools])
        COST["chat"] += r.usage.cost
        info["prompt_tokens"] = info["prompt_tokens"] or r.usage.prompt_tokens
        msg = r.choices[0].message
        if not msg.tool_calls:
            return msg.content, info
        # ✏️ TODO 1 (▶ video 2:51): Run the tool the model picked, and append its result to the messages
        msgs.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            print(f"  🔧 {call.function.name}({args})")
            if call.function.name == "get_paper":
                info["fetched"].add(args["paper_id"])
            msgs.append({"role": "tool", "tool_call_id": call.id,
                         "content": ___})
    return None, info

## 2. The task, without a skill

The system prompt is one line, and the agent gets all three tools

In [ ]:
BASE = "You are a research assistant for university students."
TASK = "Summarize 3 papers on agent memory for first-year students."

plain, plain_info = run_agent(BASE, TASK, ["search_papers", "get_paper", "word_count"])
print(f"\n{len(plain.split())} words · papers opened: {sorted(plain_info['fetched'])}\n")
print(plain[:700], "…")

## 3. Write the skill as a folder

```
skills/paper-summary/
├── SKILL.md     ← header (name, description, version, tools) + instructions, workflow and guardrails
└── example.md   ← knowledge: an example of a good answer
```

| Part of a skill (§22.1) | Where it lives |
|---|---|
| prompt: instructions, constraints, persona | the body of `SKILL.md` |
| the tools it needs | `tools:` in the header |
| knowledge | `example.md` |
| workflow | `## Workflow` |
| guardrails | `## Guardrails` |

The header between the `---` lines is YAML (called frontmatter) — several tools use this `SKILL.md` layout; the book notes there is no industry-standard schema yet (§22.5)

In [ ]:
Path("skills/paper-summary").mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile skills/paper-summary/SKILL.md
---
name: paper-summary
description: Summarize research papers from the library for students, with a citation on every sentence.
version: 1.0.0
tools: [search_papers, get_paper]
knowledge: [example.md]
---
# Paper summary

Write for first-year students: plain words, no jargon without a short explanation.

## Workflow
1. Call search_papers with the topic.
2. Call get_paper for each paper you will use. Use only what get_paper returns.
3. Compare the papers: what does each one add?
4. Write the answer.

## Guardrails
- Answer with JSON only, no other text: {"papers": ["<id>", ...], "sentences": [{"text": "...", "cite": "<id>"}, ...]}
- Every sentence cites exactly one paper id that you fetched with get_paper.
- At most 120 words in all sentences together.
- Add no claim, number or adjective that the paper record does not contain.

In [ ]:
%%writefile skills/paper-summary/example.md
# Example of a good answer (topic: tool use)
{"papers": ["2210.03629", "2302.04761"],
 "sentences": [
  {"text": "ReAct lets a model think and then act with a tool, again and again in one loop.", "cite": "2210.03629"},
  {"text": "Toolformer goes further: the model learns by itself where an API call helps.", "cite": "2302.04761"}]}

## 4. Load and activate the skill

Stage 3 of the skill lifecycle (§22.4), **activation**: put the skill's instructions, tools and knowledge into the context

In [ ]:
# ✏️ TODO 2 (▶ video 5:31): Turn the YAML header into a dictionary
def load_skill(folder):
    folder = Path(folder)
    _, header, body = (folder / "SKILL.md").read_text().split("---", 2)
    skill = ___
    skill["instructions"] = body.strip()
    skill["knowledge_text"] = "\n\n".join((folder / f).read_text() for f in skill.get("knowledge", []))
    return skill

skill = load_skill("skills/paper-summary")
print({k: skill[k] for k in ["name", "version", "tools", "knowledge"]})
print(skill["description"])

In [ ]:
# ✏️ TODO 3 (▶ video 6:02): Return the skill's own tools (not every tool)
def activate(skill, base=BASE):
    system = f"{base}\n\n{skill['instructions']}\n\n## Knowledge\n{skill['knowledge_text']}"
    return system, ___

## 5. Check the guardrails in code, and retry when they fail

Rules that code can check shouldn't be left to the model (EP04: the model often missed counting rules)
If a check fails, send the problems back for one retry — **build in recovery** (§22.3.4)

In [ ]:
def check(answer, fetched, max_words=120):
    text = (answer or "").strip().removeprefix("```json").removesuffix("```").strip()
    try:
        sentences = json.loads(text)["sentences"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return ["not JSON with a 'sentences' list"]
    # ✏️ TODO 4 (▶ video 6:44): Find sentences that cite a paper that was not fetched
    problems = [f"sentence {i} cites {s.get('cite')!r}, which was not fetched with get_paper"
                for i, s in enumerate(sentences, 1) if ___]
    words = sum(len(s.get("text", "").split()) for s in sentences)
    if words > max_words:
        problems.append(f"{words} words, more than {max_words}")
    return problems

In [ ]:
def run_skill(skill, task):
    system, tools = activate(skill)
    answer, info = run_agent(system, task, tools)
    problems = check(answer, info["fetched"])
    # ✏️ TODO 5 (▶ video 7:12): If there are problems, retry once with the list of problems
    if ___:
        print("  ⚠️ retry:", problems)
        answer, info2 = run_agent(system, f"{task}\n\nYour last answer broke these rules: {problems}. Answer again.", tools)
        info["fetched"] |= info2["fetched"]
        problems = check(answer, info["fetched"])
    return answer, info, problems

## 6. The same task, with the skill

In [ ]:
answer, info, problems = run_skill(skill, TASK)
print(f"\nchecks: {problems or 'all passed'}\n")
for s in json.loads(answer.strip().removeprefix("```json").removesuffix("```"))["sentences"]:
    print(f"[{s['cite']}] {s['text']}")

In [ ]:
print(f"{'':16}{'without skill':>15}{'with skill':>12}")
print(f"{'checks passed':16}{str(not check(plain, plain_info['fetched'])):>15}{str(not problems):>12}")
print(f"{'input tokens':16}{plain_info['prompt_tokens']:>15}{info['prompt_tokens']:>12}")
print(f"\nthe skill adds {info['prompt_tokens'] - plain_info['prompt_tokens']} tokens to every call · cost so far: ${COST['chat']:.5f}")

## Summary

| | Without a skill | With the `paper-summary` skill |
|---|---|---|
| workflow | the agent guesses (some runs opened no paper at all) | search → open every paper → compare → write |
| tools | all three | only `search_papers`, `get_paper` (tool binding) |
| format | hundreds of words of markdown, no citations | JSON, every sentence cites a paper it opened |
| cost | — | more input tokens on every model call |

A skill is **text in files** — edit the file and the behaviour changes at once, with no retraining (§22.6)

**Exercises**
1. Change the guardrail to `At most 40 words` but keep `check` at `max_words=120` — then swap them: 120 in the skill, 40 in the check, and see whether recovery kicks in
2. Add a workflow step that makes the agent use `word_count` before answering — what must change in the header too?
3. Write a second skill, `paper-quiz`, that writes one quiz question per paper, using the same `load_skill` and `run_skill`

**Next — Lab 2:** many skills in one library; the model picks from their descriptions, and we count how much context that saves